# 4a. ICA on the localizer tasks (motor and visual)

**Why.** The ICA of `01_ica` is fitted on the whole session and is used to *remove* artifacts. For the spatial filters (`04`), the supervisor asked for an ICA fitted only on a **localizer task**, one per brain area:

| Localizer | Task | Area | Window (annotations of each recording) |
|---|---|---|---|
| `motor` | bubble popping (motor baseline): the same bubbles on all six iPads, tapped | motor (mu/beta) | `TRIAL_START/motor_baseline/...` → `TRIAL_END/motor_baseline/...` |
| `visual` | instruction video, watched without playing | visual (alpha) | `DISPLAY_ONSET/video_first_frame` (photodiode) → `MARKER/code ... stop (instructions_video)` |

The component found in a localizer is then used as a spatial filter on the game trials. The filter is learned from data the analysis does not test, so choosing it cannot favour any condition.

**Input: the clean data** of `03_clean` (eye components removed, 0.1–40 Hz). Otherwise the video ICA would spend several components on eye movements.

**Same settings as `01_ica`** (`config.py`): training copy 1–40 Hz → 250 Hz, 1 s dummy epochs, joint-probability rejection at 3 SD, extended Infomax, fixed seed. Bad stretches (`BAD_amplitude`) are left out; breaks (`BAD_break`) are not, because the video lies in the break between the motor baseline and the first Solo round.

**No automatic choice.** The notebook saves the maps and property plots of each localizer ICA; the visual and motor components are chosen by eye, with the supervisor.

**Steps:** 1. load the clean data → 2. per localizer: cut the window, training copy, dummy epochs → 3. ICA → 4. save the ICA, the maps and the property plots.

In [ ]:
import os
import re

import mne
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from mne.preprocessing import ICA

import config as cfg
from utils import jointprob_keep, plot_component_maps
%matplotlib inline

## 0. Participant

In [ ]:
# execute_notebooks.py sets RT_SUBJECT
SUBJECT = os.environ.get("RT_SUBJECT", "sub-001")
SESSION = cfg.session_of(SUBJECT)
OUT_DIR = cfg.out_dir(SUBJECT)
REVIEW_DIR = cfg.OUT_ROOT / "spatial_filter_review" / SESSION / "localizer"  # PNGs for the review, one folder per session
REVIEW_DIR.mkdir(parents=True, exist_ok=True)
print(SUBJECT, f"({SESSION})", "->", OUT_DIR, "and", REVIEW_DIR)

## 1. Load the clean data

The clean data still has every annotation of the recording (trials, phase markers, photodiode onsets, `BAD_amplitude`), so the localizer windows can be found in it.

**Rank.** The clean data has fewer independent signals than channels: one less for the common average reference, one less for each interpolated channel (both already in the number of components of the `01` ICA), and one less for each component `03` removed. So the localizer ICA can have at most

rank = components of the `01` ICA − removed components

The rank estimated from the data (`mne.compute_rank`, in step 2) is printed as a check; it should be the same number.

In [ ]:
raw = mne.io.read_raw(OUT_DIR / f"{SUBJECT}_clean_raw.fif", preload=True)
ica_session = mne.preprocessing.read_ica(OUT_DIR / f"{SUBJECT}_ica.fif")
removed = pd.read_csv(OUT_DIR / f"{SUBJECT}_badcomps.csv")["component"].astype(int).tolist()
rank = ica_session.n_components_ - len(removed)
print(f"{SUBJECT}: {raw.info['sfreq']} Hz, {raw.times[-1]:.1f} s")
print(f"01 ICA: {ica_session.n_components_} components, removed in 03: {removed} -> rank of the clean data: {rank}")

## 2. Cut the localizer windows, training copy, dummy epochs

For each localizer:

1. find its start and end annotation in this recording (the first end after the start);
2. cut the window with extra data on each side (up to `LOCALIZER_MARGIN`, 10 s; only 2 s before the motor baseline, where `01` cut the data), filter 1–40 Hz, then cut the margins off, so the filter has no edge effects inside the window;
3. downsample to 250 Hz;
4. 1 s dummy epochs, skipping `BAD_amplitude`; `BAD_break` is removed first (the video lies in a break);
5. joint-probability rejection at 3 SD, as in `01_ica`.

The table shows the length of each window and how many epochs are left. With 180 s at 250 Hz there are ~45 000 samples, well above the usual minimum for ICA (about 20 × components², ~10 000 here).

In [ ]:
def annotation_times(raw, pattern):
    # Times (s from the start of the data) of the annotations matching a regular expression
    ann = raw.annotations
    offset = raw.first_time if ann.orig_time is not None else 0.0  # same convention as 01_ica
    return np.array([onset - offset for onset, d in zip(ann.onset, ann.description) if re.search(pattern, d)])


def localizer_window(raw, start_pattern, end_pattern):
    # (start, end) of a localizer task in seconds from the start of the data
    starts = annotation_times(raw, start_pattern)
    assert len(starts) > 0, f"no annotation matches {start_pattern}"
    start = starts.min()
    ends = annotation_times(raw, end_pattern)
    ends = ends[ends > start]
    assert len(ends) > 0, f"no annotation matches {end_pattern} after the start"
    return start, ends.min()


training, rows = {}, []
for name, (start_pattern, end_pattern) in cfg.LOCALIZERS.items():
    start, end = localizer_window(raw, start_pattern, end_pattern)

    # filter with a margin on each side, then cut it off. The margin is as long as the data allows, up to
    # LOCALIZER_MARGIN: 01_ica cut the data 2 s before the first trial, which is the motor baseline.
    # It must cover half the filter (MNE's 1 Hz high-pass is 3.3 s long, so 1.65 s).
    margin = min(cfg.LOCALIZER_MARGIN, start, raw.times[-1] - end)
    assert margin >= 1.65, f"{name}: only {margin:.2f} s of data around the window, less than half the filter"
    x = raw.copy().crop(tmin=start - margin, tmax=end + margin)
    x.filter(cfg.ICA_HP, cfg.ICA_LP, picks=["eeg", "eog"], verbose="error")  # 1-40 Hz, as in 01_ica
    x.crop(tmin=margin, tmax=margin + (end - start))
    x.resample(cfg.ICA_SRATE, verbose="error")

    # breaks do not apply here (the window is chosen by its own events); bad stretches still do
    x.annotations.delete(np.where(x.annotations.description == "BAD_break")[0])
    epochs = mne.make_fixed_length_epochs(x, duration=cfg.ICA_EPOCH_LENGTH, reject_by_annotation=True,
                                          preload=True, verbose="error")
    keep = jointprob_keep(epochs.get_data(picks="eeg"), cfg.ICA_PRUNE)
    training[name] = (x, epochs[np.where(keep)[0]])

    rows.append({"subject": SUBJECT, "localizer": name, "start_s": round(start, 2), "margin_s": round(margin, 2), "duration_s": round(end - start, 1),
                 "n_dummy_epochs": len(epochs), "n_rejected_epochs": int(np.sum(~keep)),
                 "n_kept_epochs": int(np.sum(keep)),
                 # relative tolerance: the clean file is stored in single precision, whose rounding noise
                 # (~1e-8 of the largest signal) would otherwise count as extra dimensions
                 "rank_from_data": mne.compute_rank(epochs[np.where(keep)[0]], rank=None, tol=1e-4,
                                                    tol_kind="relative", verbose="error")["eeg"]})

summary = pd.DataFrame(rows)
summary

## 3. ICA per localizer

Extended Infomax on the EEG channels, as in `01_ica`, with as many components as the rank of the clean data (step 1). The number of iterations is saved: if it equals the limit, the ICA stopped before it converged.

In [ ]:
localizer_icas = {}
for name, (x, epochs_ica) in training.items():
    ica = ICA(
        n_components=rank,
        method="infomax",
        fit_params=dict(extended=True),  # extended Infomax = EEGLAB runica 'extended', 1
        max_iter="auto",
        random_state=cfg.RANDOM_STATE,
    )
    ica.fit(epochs_ica, picks="eeg")
    localizer_icas[name] = ica
    print(f"{name}: {ica} | iterations: {ica.n_iter_}")

summary["n_ica_components"] = [localizer_icas[n].n_components_ for n in summary["localizer"]]
summary["n_ica_iterations"] = [localizer_icas[n].n_iter_ for n in summary["localizer"]]
summary

## 4. Save the ICAs, the maps and the property plots

Per localizer:

- the ICA: `<sub>/<sub>_<localizer>_localizer_ica.fif` (for `04`);
- all scalp maps in one PNG: `spatial_filter_review/<session>/localizer/<sub>_<localizer>_maps.png`;
- the property plot (map, time course, spectrum) of every component, on the localizer's own data: `.../localizer/<sub>_<localizer>/<sub>_<localizer>_ICA<NNN>.png`. The spectrum shows whether a component has an alpha peak (visual) or a mu/beta peak (motor).

Plus one table for the Methods: `<sub>/<sub>_localizer_ica_info.csv`.

In [ ]:
for name, ica in localizer_icas.items():
    ica.save(OUT_DIR / f"{SUBJECT}_{name}_localizer_ica.fif", overwrite=True)

    figure = plot_component_maps(ica, f"{SUBJECT}: ICA on the {name} localizer")
    figure.savefig(REVIEW_DIR / f"{SUBJECT}_{name}_maps.png", dpi=100)
    plt.close(figure)

    component_dir = REVIEW_DIR / f"{SUBJECT}_{name}"
    component_dir.mkdir(exist_ok=True)
    for old in component_dir.glob("*.png"):  # plots of an earlier run
        old.unlink()
    x, epochs_ica = training[name]
    for comp in range(ica.n_components_):
        figure = ica.plot_properties(epochs_ica, picks=[comp], psd_args=dict(fmax=40), show=False)[0]
        figure.suptitle(f"{SUBJECT} {name} localizer ICA{comp:03d}", fontsize=9, y=1.04)
        figure.savefig(component_dir / f"{SUBJECT}_{name}_ICA{comp:03d}.png", dpi=100, bbox_inches="tight")
        plt.close(figure)

summary["reference"] = cfg.REFERENCE or "mastoids"
summary.to_csv(OUT_DIR / f"{SUBJECT}_localizer_ica_info.csv", index=False)
print("Saved to", OUT_DIR, "and", REVIEW_DIR)
summary

## Next

Choose one visual component (from the `visual` localizer ICA) and one motor component (from the `motor` localizer ICA) per participant, with the supervisor. Then `04` applies those localizer ICAs to the clean game data and computes the envelopes (to be adapted: `SPATIAL_FILTER_CHOICE` must say which ICA a component comes from).